# Do checks and quality flags for extracted data
1. Load data
2. Convert to correct format (numeric for numerical columns)
3. Sanity checks

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import geopy as gpy
import time
import itertools
import regex as re
from matplotlib import pyplot as plt
from src.data import *
from src.plot_functions import *
from src.post_process_functions import *
from src.geocoding import *
from src.hazard_def import *
from src.impact_def import *
from src.sanity_checks import *

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /Users/lseverino/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/lseverino/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
#load data (model)
models = ["meta-llama_llama-4-scout-17b-16e-instruct", "llama-3.1-8b-instant", "llama-3.3-70b-versatile"]
filename_in = f"labelled_reports_{models[2]}_v250925"#"labelled_reports_impacts_all_v080925" 'llm_response_impact_labelled_reports_test_multiprompt_continue_v050925_21rep_meta-llama_llama-4-scout-17b-16e-instruct'
data_path = DATA_OUT_LLMS #DATA_LABELLED DATA_OUT_LLMS  (depending on whether we want to process the LLM output or the labelled data)

## Load data
response_df = pd.read_csv(data_path / (filename_in + ".csv"))


In [4]:
#get rid of nans
response_df = response_df.dropna(subset=["nathaz_text"]) if "nathaz_text" in response_df.columns else response_df

In [5]:
response_df

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,country,location,locationAnnotation,...,endDay,dateAnnotation,hazards,hazardsAnnotation,appealCode,country_kw,reportDate,reportLink,disasterType,nathaz_text
0,Affected People,326788.0,NaN,NaN,exact,people,"['People Affected: 326,788 people']",['Pakistan'],"['Balochistan', 'Khyber Pakhtunkhwa', 'Sindh']","['Targeted Regions: Balochistan, Khyber Pakhtu...",...,31.0,['Pakistan endured an exceptionally intense mo...,['Flood'],['Pakistan endured an exceptionally intense mo...,MDRPK026,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...
1,Injured People,584.0,NaN,NaN,exact,people,['The monsoon season caused 306 fatalities and...,['Pakistan'],"['Balochistan', 'Sindh', 'Khyber Pakhtunkhwa',...",['The monsoon season caused 306 fatalities and...,...,NaN,['Pakistan endured an exceptionally intense mo...,"['Flood', 'Mass movement']",['Pakistan endured an exceptionally intense mo...,MDRPK026,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...
2,Human Deaths,306.0,NaN,NaN,exact,people,['The monsoon season caused 306 fatalities and...,['Pakistan'],"['Balochistan', 'Sindh', 'Khyber Pakhtunkhwa',...",['Pakistan endured an exceptionally intense mo...,...,NaN,['Pakistan endured an exceptionally intense mo...,['Flood'],['Pakistan endured an exceptionally intense mo...,MDRPK026,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...
3,Residential Buildings,20653.0,NaN,NaN,exact,homes,['The monsoon season caused 306 fatalities and...,['Pakistan'],"['Balochistan', 'Sindh', 'Khyber Pakhtunkhwa']","['In Balochistan, floods affected more than 17...",...,NaN,['Pakistan endured an exceptionally intense mo...,['Flood'],['Pakistan endured an exceptionally intense mo...,MDRPK026,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...
4,Road Infrastructure,40.0,NaN,NaN,exact,bridges,['The monsoon season caused 306 fatalities and...,['Pakistan'],NaN,['The monsoon season caused 306 fatalities and...,...,NaN,['Pakistan endured an exceptionally intense mo...,['Flood'],['Pakistan endured an exceptionally intense mo...,MDRPK026,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
445,Access to Food,269543.0,NaN,NaN,exact,Swiss francs (CHF),"['Appeal Coverage to date: 13 % (269,543 CHF)']",['Guatemala'],"['El Quiché', 'Western Temperate Highlands']",['Host National Society presence: Guatemalan R...,...,4.0,"['Operation start date: 3 February 2016', 'Tim...",['Drought'],['Emergency Appeal Operations Update Guatemala...,MDRGT009,Guatemala,2016-11-14,https://adore.ifrc.org/Download.aspx?FileId=15...,Drought,"['P a g e | 1 Operation Update no.', '2 Operat..."
446,Human Health and Wellbeing,50.0,NaN,NaN,exact,training sessions in hygiene promotion,['\uf0fc From 130 to 50 training sessions in h...,['Guatemala'],['El Quiché'],['Host National Society presence: Guatemalan R...,...,4.0,['The emergency appeal in Quiché was launched ...,['Drought'],['According to the Food Security Outlook Updat...,MDRGT009,Guatemala,2016-11-14,https://adore.ifrc.org/Download.aspx?FileId=15...,Drought,"['P a g e | 1 Operation Update no.', '2 Operat..."
447,Access to Food,13.0,NaN,NaN,exact,%,"['Appeal Coverage to date: 13 % (269,543 CHF)']",['Guatemala'],"['El Quiché', 'Western Temperate Highlands']",['Host National Society presence: Guatemalan R...,...,4.0,"['Operation start date: 3 February 2016', 'Ope...",['Drought'],['According to the Food Security Outlook Updat...,MDRGT009,Guatemala,2016-11-14,https://adore.ifrc.org/Download.aspx?FileId=15...,Dro

In [6]:
#convert numerical columns
#convert numerical columns
num_cols = ["impactValue", "impactValueMin", "impactValueMax","startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]
list_cols = ["country","location", "hazards", "valueAnnotation", "locationAnnotation", "dateAnnotation", "hazardsAnnotation", "annotation"]
list_cols = [key for key in list_cols if key in response_df.columns]
response_df_proc = format_output(response_df, num_cols=num_cols, list_cols=list_cols)




In [7]:
#add iso3
response_df_proc["country_iso3"] = response_df_proc["country"].apply(country_name_to_iso3)
response_df_proc["country_iso3_kw"] = response_df_proc["country_kw"].apply(country_name_to_iso3) if "country_kw" in response_df_proc.columns else None

In [8]:
#format units
from spacy.lang.en import English
from spacy.lang.punctuation import TOKENIZER_PREFIXES, TOKENIZER_SUFFIXES, TOKENIZER_INFIXES
from spacy.lang.en import TOKENIZER_EXCEPTIONS
from spacy.tokenizer import Tokenizer
from spacy.util import compile_prefix_regex, compile_suffix_regex, compile_infix_regex

## Sanity checks
1. ValueInText: ImpactValue should be in the original text
2. MaxPop: ImpactValue with “people” unit should be smaller than country’s population
3. OrigSentence: Annotation sentence should be present in the original text
4. UnknownImpType: Inferred impactType must be in the allowed list
5. UnknownHaz: Inferred hazardType must be in allowed list
6. Location partially undefined. 


In [9]:
response_df_proc.impactUnit.value_counts()

impactUnit
people                                    111
houses                                     38
CHF                                        26
households                                 19
homes                                      15
                                         ... 
males                                       1
females                                     1
% of buildings                              1
% of homes                                  1
administrative and logistics assistant      1
Name: count, Length: 80, dtype: int64

In [ ]:
response_df_exploded = response_df_proc.explode("hazards", ignore_index=True)



In [31]:
response_df_exploded.hazards.value_counts()

hazards
Flood                       296
Tropical storm               82
Drought                      69
Mass movement                40
Convective storm             38
Other storm                  27
Epidemic                     22
Conflict                     18
Extreme cold temperature     16
Wildfire                     14
Volcanic activity            10
Wave action                   2
Name: count, dtype: int64

In [28]:
response_df_exploded[response_df_exploded["appealCode"]=="MDRS2001"]

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,country,location,locationAnnotation,...,hazards,hazardsAnnotation,appealCode,country_kw,reportDate,reportLink,disasterType,nathaz_text,country_iso3,country_iso3_kw
1612,Affected People,25000.0,NaN,NaN,exact,people,There is also an urgent need for disease preve...,Barbados,Guria,"Meanwhile, in Tamanrasset, 100 families from N...",...,Flood,The intensification and recurrence of rains st...,MDRS2001,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic...",Unknown,PAN
1613,Affected People,25000.0,NaN,NaN,exact,people,"The destruction of 1,865 latrines poses a seve...",Grenada,Adjara,"Tragically, the floods have claimed five lives...",...,Mass movement,Series of floods have been recorded since Augu...,MDRS2001,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic...",Unknown,PAN
1614,Affected People,25000.0,NaN,NaN,exact,people,There is also an urgent need for disease preve...,Jamaica,Imereti,"The most affected areas include Béchar, Elbaya...",...,Flood,The peak of the floods was recorded on August ...,MDRS2001,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic...",Unknown,PAN
1615,Affected People,25000.0,NaN,NaN,exact,people,"The destruction of 1,865 latrines poses a seve...",Saint Vincent and the Grenadines,Mtskheta-Mtianeti,"In Béchar, the number of displaced families su...",...,Mass movement,Cameroon's Far North region has been experienc...,MDRS2001,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic...",Unknown,PAN
1616,Affected People,25000.0,NaN,NaN,exact,people,There is also an urgent need for disease preve...,Barbados,Chokhatauri,"In Elbayadh, 60 families have been affected, a...",...,Flood,The intensification and recurrence of rains st...,MDRS2001,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic...",Unknown,PAN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1683,Other Economic Activity & Livelihood Production,400000.0,NaN,NaN,exact,CHF,"In Bweramule Sub-County, 714 households were a...",Jamaica,Mtskheta-Mtianeti,Cameroon's Far North region has been experienc...,...,Mass movement,Cameroon's Far North region has been experienc...,MDRS2001,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic...",Unknown,PAN
1684,Other Economic Activity & Livelihood Production,400000.0,NaN,NaN,exact,CHF,"In Bweramule Sub-County, 714 households were a...",Saint Vincent and the Grenadines,Chokhatauri,"The most affected districts are Blangoua, Mack...",...,Flood,The intensification and recurrence of rains st...,MDRS2001,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic...",Unknown,PAN
1685,Other Economic Activity & Livelihood Production,400000.0,NaN,NaN,exact,CHF,Butungama Sub-County saw 578 households affect...,Barbados,Gogole District,"Notably, in the Diamaré division, where Ndouko...",...,Mass movement,Series of floods have been recorded since Augu...,MDRS2001,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic...",Unknown,PAN
1686,Other Economic Activity & Livelihood Production,600000.0,NaN,NaN,exact,CHF,Butungama Sub-County saw 578 households affect...,Grenada,Ozurgeti,"The most affected districts are Blangoua, Mack...",...,Flood,Cameroon's Far North region has been experienc...,MDRS2001,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic..."

In [27]:
response_df_proc[response_df_proc["appealCode"]=="MDRS2001"]

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,country,location,locationAnnotation,...,country_kw,reportDate,reportLink,disasterType,nathaz_text,country_iso3,country_iso3_kw,value_in_text,pop_cntry_check,unknown_impactSubtype
257,Affected People,2.500000e+04,NaN,NaN,exact,people,"[Number of people being assisted: 25,000]","[Barbados, Grenada, Jamaica, Saint Vincent and...","[Carriacou, Petit Martinique, Union Island, Cl...","[Although 55 homes suffered minor damages, the...",...,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic...",Unknown,PAN,True,True,False
258,Residential Buildings,5.500000e+01,NaN,NaN,exact,homes,"[Although 55 homes suffered minor damages, the...",[Barbados],[],"[Although 55 homes suffered minor damages, the...",...,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic...",Unknown,PAN,True,NaN,False
259,Residential Buildings,9.800000e+01,NaN,NaN,exact,% of buildings,"[In Grenada, more than 1,600 people were force...",[Grenada],"[Carriacou, Petit Martinique]","[In Grenada, more than 1,600 people were force...",...,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic...",Unknown,PAN,True,NaN,False
260,Residential Buildings,9.000000e+01,NaN,NaN,exact,% of homes,[Saint Vincent and the Grenadines experienced ...,[Saint Vincent and the Grenadines],[Union Island],[Saint Vincent and the Grenadines experienced ...,...,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic...",Unknown,PAN,True,NaN,False
261,Agricultural Infrastructure,1.000000e+09,NaN,NaN,exact,USD,[The agricultural sector alone suffered losses...,[Jamaica],"[Clarendon, St. Elizabeth, St. Thomas, Manches...",[Jamaica experienced widespread damage as Bery...,...,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic...",Unknown,PAN,True,NaN,True
262,Other Economic Activity & Livelihood Production,2.000000e+02,NaN,NaN,exact,vessels,"[Although 55 homes suffered minor damages, the...",[Barbados],[],"[Although 55 homes suffered minor damages, the...",...,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic...",Unknown,PAN,True,NaN,True
263,Access to Healthcare,NaN,NaN,NaN,NaN,NaN,[The destruction of homes and critical infrast...,[Saint Vincent and the Grenadines],[Union Island],[Saint Vincent and the Grenadines experienced ...,...,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic...",Unknown,PAN,NaN,NaN,False
264,Access to Food,NaN,NaN,NaN,NaN,NaN,[The agricultural sector alone suffered losses...,[Jamaica],"[Clarendon, St. Elizabeth, St. Thomas, Manches...",[Jamaica experienced widespread damage as Bery...,...,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic...",Unknown,PAN,NaN,NaN,False
265,"Access to Water, Sanitation, and Hygiene",NaN,NaN,NaN,NaN,NaN,"[This displacement, combined with damaged wate...","[Barbados, Grenada, Jamaica, Saint Vincent and...",[],[The destruction of homes and critical infrast...,...,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE Barbados, Grenada, Jamaic...",Unknown,PAN,NaN,NaN,False
266,Human Health and Wellbeing,NaN,NaN,NaN,NaN,NaN,[The psychological impact on survivors is also...,"[Barbados, Grenada, Jamaica, Saint Vincent and...","[Carriacou, Petit Martinique, Union Island, Cl...","[Although 55 homes suffered minor damages, the...",...,Panama,2024-08-06,https://adore.ifrc.org/Download.aspx?FileId=83...,Hurricane,"['1 OPERATION UPDATE B

In [12]:
#value in original text
response_df_proc = flag_value_in_text(response_df_proc)

In [13]:
response_df_proc.value_in_text.value_counts()

value_in_text
True     323
False     19
Name: count, dtype: int64

In [23]:
response_df_proc[response_df_proc["value_in_text"] == False][["impactValue", "impactUnit", "valueAnnotation", "nathaz_text"]]

,impactValue,impactUnit,valueAnnotation,nathaz_text
12,15500.0,people,"[Of these, more than 15,000 were completely de...",['DREF Final Report Pakistan Flood August 2024...
44,57895.0,people,[The cumulative cases jumped from 2895 cases (...,['DREF Operational Update Sudan Floods 2024 Di...
47,52099.0,houses,[OCHA reports indicates that floods as of Marc...,['DREF Operational Update Sudan Floods 2024 Di...
49,17118.0,people,[Main epicenters as of 10.03.2025 included Whi...,['DREF Operational Update Sudan Floods 2024 Di...
81,61265.0,people,"[Number of people being assisted: 61,165 peopl...",['1 OPERATION UPDATE Mozambique | Drought Emer...
85,0.9,million CHF,[CHF 6 million Federation-wide DREF amount ini...,['1 OPERATION UPDATE Mozambique | Drought Emer...
159,51000.0,people,"[Overall, 14 districts experienced flooding an...",['DREF Final Report Rwanda - Floods and Landsl...
162,5293.0,houses,"[The assessment also revealed that 4,933 were ...",['DREF Final Report Rwanda - Floods and Landsl...
174,7119.0,people,"[More than 7,000 people were located in the of...",['DREF Final Report Rwanda - Floods and Landsl...
188,98997.0,CHF,"[Operation budget: 99,897 CHF]",['DREF operation Operation n° MDRTN010 Date of...


In [24]:
['DREF Final Report Pakistan Flood August 2024 Non-food items distribution by PRCS to assist communities affected by flooding in districts of Sindh, Balochistan and Khyber Pakhtunkhwa, facilitated by the support of IFRC.', '(Photo: PRCS NHQ) Appeal: MDRPK026 Total DREF Allocation: CHF 440,161 Crisis Category: Yellow Hazard: Flood Glide Number: FL-2024-000166-PAK People Affected: 326,788 people People Targeted: 5,600 people People Assisted: 13,748 people Event Onset: Sudden Operation Start Date: 14-09-2024 Operational End Date: 31-12-2024 Total Operating Timeframe: 3 months Targeted Regions: Balochistan, Khyber Pakhtunkhwa, Sindh The major donors and partners of the IFRC-DREF include the Red Cross Societies and governments of Australia, Austria, Belgium, Britain, China, Czech, Canada, Denmark, German, Ireland, Italy, Japan, Luxembourg, Liechtenstein, Malta, Norway, Spain, Sweden, Switzerland, Thailand, and the Netherlands, as well as DG ECHO, Mondelez Foundation, and other corporate and private donors.', 'The IFRC, on behalf of the National Society, would like to thank all for their generous contributions.', 'Page 1 / 20 Description of the Event Map of target areas of response.', '(Map: IFRC, IM) Date of event 01-09-2024 What happened, where and when?', 'Pakistan endured an exceptionally intense monsoon season starting in June 2024, resulting in significant infrastructure degradation, numerous casualties, and widespread injuries.', 'In August 2024, rainfall increased by 137 per cent compared to historical averages, with regions such as Sindh experiencing an unprecedented 318 per cent increase.', 'The regional impacts were severe and multifaceted.', 'In Balochistan, rainfall levels surged by 239 per cent, leading to flash floods and landslides that affected over 13 districts in the province.', 'Emergency declarations were issued in eight districts, impacting more than 109,602 individuals.', 'Sindh experienced acute urban flooding, particularly in cities like Badin, Dadu, and Jacobabad, displacing approximately 9,500 residents and causing extensive damage to houses and critical infrastructure.', 'Punjab, on the other hand, faced riverine flooding and hill torrents, resulting in significant agricultural losses.', 'In Khyber Pakhtunkhwa (KP), heavy precipitation triggered flash floods, exacerbating damage to residential and public infrastructure.', 'The monsoon season caused 306 fatalities and 584 injuries, alongside substantial infrastructural damage, including 40 bridges and 20,653 homes overall.', 'As a result, urgent humanitarian needs emerged, encompassing access to Water, Sanitation, and Hygiene (WASH) services, the provision of shelter and hygiene kits, the establishment of medical camps to mitigate disease outbreaks, and the provision of cash assistance to impacted households and agricultural workers.', 'Additionally, the stagnant water in low-lying areas and the damage to infrastructure posed significant health risks for the affected populations, complicating access to essential health services [1].', 'References: [1] Page 2 / 20 PRCS staff conducting a session on health education in district Badin (Photo: PRCS NHQ) Local residents of district Sohbatpur collecting water from the PRCS installed SETA plant (Photo: PRCS NHQ) PRCS WASH team installing SETA plant in district Sohbatpur, Balochistan (Photo: PRCS NHQ) PRCS staff providing health assistance to flood affected people in district Badin (Photo: PRCS NHQ) Scope and Scale The National Disaster Management Authority (NDMA) and the Pakistan Meteorological Department (PMD) issued alerts indicating that several regions, including Balochistan, KP, Sindh, Punjab, Azad Jammu and Kashmir (AJK), and Gilgit-Baltistan (GB), were particularly vulnerable.', 'The threat of Glacial Lake Outburst Floods (GLOFs) was also notable in KP and GB, exacerbated by the rising temperatures in the region.', 'The extreme precipitation, combined with unusually high temperatures, accelerated snowmelt in KP, AJK, and GB, resulting in catastrophic floods and landslides that impacted many regions.', 'Areas such as Jacobabad, Naushero Feroz, Ghotki, Sukkur, Sanghar, Dadu, Shaheed Benazirabad, and Kashmore faced heavy rains that caused substantial damage to homes and infrastructure.', 'Impact Assessment The impacts of the monsoon season were profound, with damage reports indicating: • Widespread destruction of homes and infrastructure • Disruption of essential water supplies, critically affecting communities • Loss of life, necessitating urgent humanitarian intervention In regions such as KP and Sindh, particularly in Chitral and Badin, there was an urgent demand for essential resources, including multipurpose cash grants, clean water, shelter kits, and hygiene supplies.', 'In response to this escalating crisis, the Pakistan Red Crescent Society (PRCS) actively engaged in disaster response, coordinating efforts in affected areas such as Chitral, Badin, and Sohbatpur.', 'PRCS conducted rapid needs assessments in collaboration with local authorities to evaluate the situation and mobilize resources effectively.', 'Therefore, to address the immediate needs arising from the monsoon crisis, PRCS launched a DREF operation in partnership with the IFRC.', 'This operation focused on: • Distribution of Non-Food Items (NFIs), clean water, shelter kits, and hygiene kits • Deployment of Mobile Health Teams (MHTs) to provide essential medical support Page 3 / 20 • Restoring basic services, including water filtration and health services Additionally, PRCS enhanced its operational capacity by training staff and volunteers on necessary skills to ensure effective and sustained support for ongoing relief efforts.', 'The scale of the crisis necessitated comprehensive and coordinated interventions to help mitigate the impact of the disaster and support affected communities in their recovery.', 'Source Information Source Name Source Link 1.', 'UNOCHA 2.', 'Flash Updates#8 3.', 'National disaster Management Authority National Society Actions Have the National Society conducted any intervention additionally to those part of this DREF Operation?', 'Needs (Gaps) Identified Shelter Housing And Settlements The monsoon floods and heavy rainfall caused extensive damage to over 37,000 houses across the country, as reported in NDMA update No.', '107.', 'Of these, more than 15,000 were completely destroyed, while the remaining structures sustained partial damage.', 'In District Chitral, an assessment conducted by an international organization highlighted shelter as a critical need, revealing damage to over 700 houses.', 'In Balochistan, floods affected more than 17,000 homes, while in Sindh, over 7,700 houses were damaged.', 'The assessment report underscored the urgent requirement for shelter assistance in the flood-affected areas.', 'In response to this situation, the government took swift action by dispatching 34,430 tents to the relevant PDMAs.', 'This initiative aimed to provide immediate shelter and support to those affected, ensuring that displaced families had access to basic necessities during a challenging time [2].', 'Although the government provided tents in some of the flood-affected locations, a significant need for additional support remained.', 'In response, the national society prioritized assistance for the most vulnerable families, distributing 800 tents and 800 kitchen sets.', 'This intervention not only provided immediate relief but also enabled affected families to stay safe while protecting their privacy and dignity as they worked to reconstruct or repair their homes following the devastation caused by the monsoon rains and floods.', 'Reference: [2] Health The recent floods severely impacted communities, inundating areas and contaminating water sources, which significantly increased the risk of disease outbreaks.', 'The situation led to major health concerns, including the spread of waterborne diseases such as diarrhea, giardiasis, and dysentery, as well as an increase in vector-borne illnesses like malaria and dengue fever.', 'Stagnant floodwaters created breeding grounds for disease-carrying vectors, while damaged sanitation facilities contributed to poor hygiene conditions.', 'In response, PRCS successfully deployed a Mobile Health Unit (MHU) in the remote district of Badin, Sindh, for one month.', 'The unit provided uninterrupted healthcare services to affected populations, addressing immediate health concerns and reducing the burden on overwhelmed health facilities.', 'The MHU was equipped with essential medical supplies, including basic first aid, mobile clinics, and medications.', 'In addition, flood-affected communities were educated on hygiene, disease prevention, and maternal-child health.', 'To mitigate the spread of vector-borne diseases, PRCS distributed 1,600 Long-Lasting Insecticide-treated Nets (LLINs) among 800 Page 5 / 20 vulnerable families.', 'This intervention significantly contributed to reducing the risk of malaria and dengue transmission, offering essential protection to families as they worked towards recovery.', 'Through these targeted health interventions, PRCS effectively addressed urgent medical needs, strengthened disease prevention measures, and enhanced community resilience in the aftermath of the floods.', 'Health needs for 800 of the most affected Households (HHs) were identified, with each family receiving one hygiene kit and two Long-Lasting Insecticidal Nets (LLINs).', 'The hygiene kits were deemed necessary to help mitigate the risk of waterborne diseases by promoting safe hygiene practices, while the distribution of LLINs was essential for preventing the spread of vector-borne diseases.', 'Water, Sanitation And Hygiene In the flood-affected areas across all provinces, the absence of a proper sewerage system exacerbated the challenges related to water, sanitation, and hygiene (WASH).', 'An assessment conducted by the National Society identified an urgent need for drinking water in Sohbatpur district, Balochistan, where floodwaters had contaminated existing water sources.', 'To address this, PRCS distributed two jerry cans to targeted households for water collection and storage.', 'In addition, a SETA Water Treatment and Filtration Plant was deployed in Sohbatpur district, providing targeted households with potable water on a daily basis for one month.', 'Furthermore, hygiene promotion sessions were conducted in Sohbatpur district to enhance awareness and encourage safe hygiene practices.', 'Protection, Gender And Inclusion There was a need for a detailed needs assessment in the flood-affected areas of Balochistan, Sindh, and KP, with SADDD data collection.', 'PRCS was also aware of the importance of deploying gender-balanced volunteer teams at every stage of the operation, including assessments, distributions, awareness activities, and Post-Distribution Monitoring (PDM), to ensure that the needs of the most marginalized communities—including women, children, the elderly, religious minorities, and other vulnerable groups—were addressed within the affected communities.', 'A one-day Protection, Gender, and Inclusion (PGI) training session was identified as necessary on PGI minimum standards (DAPS) for the staff and volunteers involved in the operation.', 'It was recognized that PGI minimum standards needed to be ensured across all sectors during the response through regular orientations and technical support.', 'Community Engagement And Accountability The Community Engagement and Accountability (CEA) needs, including sharing of selection criteria, establishing effective communication channels, and ensuring active community participation, were identified at the planning phase.', 'The active involvement of the community in the planning, implementation, and monitoring phases was deemed essential to align interventions with local customs and needs while equipping communities for self-recovery.', 'To achieve this, PRCS recognized the importance of organizing regular community meetings and involving local volunteers, along with implementing transparent accountability mechanisms to ensure the quality and effectiveness of the DREF response operation.', "Environment Sustainability PRCS was committed to ensuring environmental sustainability throughout the flood response operation while adhering to the fundamental principle of 'Do No Harm.'", 'In this context, this commitment extended to employing eco-friendly practices in relief efforts, minimizing the ecological footprint of the operations, and prioritizing the preservation of local ecosystems.', 'Page 6 / 20 Operational Strategy Overall objective of the operation This IFRC-DREF operation aimed to support 5,600 people in flood-affected areas by providing family tents, kitchen sets, Mobile Health Units (MHUs), LLINs, a water treatment plant, and jerry cans, while also ensuring CEA and PGI in District Chitral Upper in KP, Sohbatpur in Balochistan, and District Badin in Sindh for three months.']

['DREF Final Report Pakistan Flood August 2024 Non-food items distribution by PRCS to assist communities affected by flooding in districts of Sindh, Balochistan and Khyber Pakhtunkhwa, facilitated by the support of IFRC.',
 '(Photo: PRCS NHQ) Appeal: MDRPK026 Total DREF Allocation: CHF 440,161 Crisis Category: Yellow Hazard: Flood Glide Number: FL-2024-000166-PAK People Affected: 326,788 people People Targeted: 5,600 people People Assisted: 13,748 people Event Onset: Sudden Operation Start Date: 14-09-2024 Operational End Date: 31-12-2024 Total Operating Timeframe: 3 months Targeted Regions: Balochistan, Khyber Pakhtunkhwa, Sindh The major donors and partners of the IFRC-DREF include the Red Cross Societies and governments of Australia, Austria, Belgium, Britain, China, Czech, Canada, Denmark, German, Ireland, Italy, Japan, Luxembourg, Liechtenstein, Malta, Norway, Spain, Sweden, Switzerland, Thailand, and the Netherlands, as well as DG ECHO, Mondelez Foundation, and other corporate an

In [15]:
#impacted people must be less than population
country_pop = pd.read_csv(DATA_PATH / ("API_SP.POP.TOTL_DS2_en_csv_v2_131993/"+"API_SP.POP.TOTL_DS2_en_csv_v2_131993.csv"),sep=',', header=2)
country_pop = country_pop.dropna(how="all",axis=1)

response_df_proc = pop_cntry_check(response_df_proc, country_pop)

 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023


In [16]:
response_df_proc.pop_cntry_check.value_counts()

pop_cntry_check
True    111
Name: count, dtype: int64

In [17]:
response_df_proc[response_df_proc["pop_cntry_check"] == False]

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,country,location,locationAnnotation,...,appealCode,country_kw,reportDate,reportLink,disasterType,nathaz_text,country_iso3,country_iso3_kw,value_in_text,pop_cntry_check


In [18]:
# check if impact are in list
def flag_impactSubtype(extracted_data, impact_list):
    def check_imp(x):
        return x["impactSubtype"] not in impact_list
    extracted_data["unknown_impactSubtype"] = np.nan
    extracted_data["unknown_impactSubtype"] = extracted_data.apply(check_imp, axis=1)
    return extracted_data

impact_list = impactSubtype_list
response_df_proc = flag_impactSubtype(response_df_proc, impact_list)
response_df_proc.unknown_impactSubtype.value_counts()

unknown_impactSubtype
False    389
True      61
Name: count, dtype: int64

In [19]:
response_df_proc[response_df_proc["unknown_impactSubtype"] == True]

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,country,location,locationAnnotation,...,country_kw,reportDate,reportLink,disasterType,nathaz_text,country_iso3,country_iso3_kw,value_in_text,pop_cntry_check,unknown_impactSubtype
34,Mobility and Access to Transport,NaN,NaN,NaN,NaN,NaN,"[The floods rendered roads impassable, further...",[Sudan],"[Atbara-Abu Hamad Road, Dongola-Halfa Road, Su...","[The floods rendered roads impassable, further...",...,Sudan,2025-03-17,https://go-api.ifrc.org/api/downloadfile/90654...,Flood,['DREF Operational Update Sudan Floods 2024 Di...,Unknown,SDN,NaN,NaN,True
72,Mobility and Access to Transport,NaN,NaN,NaN,NaN,NaN,"[Many villages remain isolated, thousands of p...",[Georgia],"[Guria, Adjara, Imereti, Mtskheta-Mtianeti, Ch...","[Many villages remain isolated, thousands of p...",...,Georgia,2025-03-16,https://go-api.ifrc.org/api/downloadfile/90652...,Extreme Winter Condition,['DREF Operation Georgia: Heavy Snowfall 2025 ...,Unknown,GEO,NaN,NaN,True
73,Other Economic Activity & Livelihood Production,NaN,NaN,NaN,NaN,NaN,[The closure of local markets due to power out...,[Georgia],"[Guria, Adjara, Imereti, Mtskheta-Mtianeti, Kh...",[The closure of local markets due to power out...,...,Georgia,2025-03-16,https://go-api.ifrc.org/api/downloadfile/90652...,Extreme Winter Condition,['DREF Operation Georgia: Heavy Snowfall 2025 ...,Unknown,GEO,NaN,NaN,True
74,Agricultural Infrastructure,NaN,NaN,NaN,NaN,NaN,[Greenhouses and irrigation systems have colla...,[Georgia],"[Guria, Adjara, Imereti, Mtskheta-Mtianeti]",[The unprecedented snowfall has led to severe ...,...,Georgia,2025-03-16,https://go-api.ifrc.org/api/downloadfile/90652...,Extreme Winter Condition,['DREF Operation Georgia: Heavy Snowfall 2025 ...,Unknown,GEO,NaN,NaN,True
83,Other Economic Activity & Livelihood Production,5.0,NaN,NaN,exact,million CHF,[Funding requirements (CHF): CHF 5 million thr...,[Mozambique],"[southern and central Mozambique, northern reg...",[Mozambique is currently experiencing severe e...,...,Mozambique,2024-10-02,https://adore.ifrc.org/Download.aspx?FileId=83...,Drought,['1 OPERATION UPDATE Mozambique | Drought Emer...,Unknown,MOZ,True,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427,Other Economic Activity & Livelihood Production,74000000.0,NaN,NaN,exact,CHF,"[Furthermore, damage to agriculture sector amo...",[Philippines],"[Aurora, Catanduanes, Nueva Vizcaya]","[Furthermore, damage to agriculture sector amo...",...,Philippines,2017-05-31,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['DREF n° MDRPH021 GLIDE n° TC-2016-000108-PHL...,Unknown,PHL,True,NaN,True
428,Other Economic Activity & Livelihood Production,4600000.0,NaN,NaN,exact,CHF,[while damage to infrastructure stands at arou...,[Philippines],"[Aurora, Catanduanes, Nueva Vizcaya]","[Furthermore, damage to agriculture sector amo...",...,Philippines,2017-05-31,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['DREF n° MDRPH021 GLIDE n° TC-2016-000108-PHL...,Unknown,PHL,True,NaN,True
429,Other Economic Activity & Livelihood Production,169011.0,NaN,NaN,exact,CHF,"[Overall operation budget: CHF 169,011]",[Philippines],"[Aurora, Catanduanes, Nueva Vizcaya]",[The Philippine Red Cross worked with Internat...,...,Philippines,2017-05-31,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['DREF n° MDRPH021 GLIDE n° TC-2016-000108-PHL...,Unknown,PHL,True,NaN,True
430,Other Economic Activity & Livelihood Production,17333.0,NaN,NaN,exact,CHF,"[The balance of CHF 17,333 will be returned to...",[Philippines],[],"[The balance of CHF 17,333 will be returned to...",...,Philippines,2017-05-31,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['DREF n° MDRPH021 GLIDE n° TC-2016-000108-PHL...,Unknown,PHL,True,NaN,True


In [20]:
# save
#savename = "flaged_" + res_savename
#response_df_proc.to_csv(DATA_OUT_LLMS + savename, index=False)